# Interpret the original Picbreeder skull

This is the **human-evolved Picbreeder CPPN**, not the separately SGD-trained imitation. The original graph has 23 nodes. The authors convert it to an equivalent layered MLP by adding identity-copy neurons, without training it again. Flattening just stores that model's parameters as one vector. Their named weight-sweep indices refer to that vector.

Here we evaluate the original graph with the authors' input and activation conventions. The four example weights have been mapped back to original connections and checked against the published layered parameter arrays. The viewer uses compact local node IDs; the example table below retains the original branch/node identifiers.

The reference is bundled and works offline with the project's normal dependencies. It does not change our evolution settings. Sources: [paper, Appendix B](https://arxiv.org/html/2505.11581v1#A2), [authors' repository](https://github.com/akarshkumar0101/fer/tree/67c270c197d8243435d6a9ad036e26c9c5e3712d), and the provenance README under `src/automated_picbreeder/reference_data/skull/`.


In [6]:
from pathlib import Path
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from PIL import Image

from automated_picbreeder.cppn import png_bytes
from automated_picbreeder.interpretability import trace_network, sweep, replay_experiment
from automated_picbreeder.interpretability_notebook import CPPNInspector
from automated_picbreeder.picbreeder_reference import load_skull, REFERENCE_DIR

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
network = load_skull(size=256)
baseline = trace_network(network)


## 1. Check which network we loaded

This graph receives horizontal position, vertical position, **1.4 × radius**, and a **constant 1** input. It uses the original signed Gaussian and unscaled sine, rather than our NEAT-Python activation definitions. The original `ink` output supplies brightness; constant-zero hue and saturation outputs reproduce the authors' grayscale-to-HSB adaptation.

The copied published PNG is included as a visual reference. Our direct graph evaluation matches an independent execution of the published layered weights within the tested numerical tolerance. It is **not pixel-identical to the repository PNG**: at 256 × 256 the measured RGB difference is approximately 0.63/255 on average, at most 12/255. The cause of that PNG discrepancy has not been established; the computation checks and the visual-reference check are distinct.


In [8]:
published = np.asarray(Image.open(REFERENCE_DIR / "published.png").convert("RGB"))
display(widgets.HBox([
    widgets.VBox([widgets.Label("Imported evolved CPPN"), widgets.Image(value=png_bytes(baseline.rgb), width=256)]),
    widgets.VBox([widgets.Label("Authors' published PNG"), widgets.Image(value=png_bytes(published), width=256)]),
]))
difference = np.abs(baseline.rgb.astype(int) - published.astype(int))
print(f"PNG difference: mean {difference.mean():.3f}/255; maximum {difference.max()}/255")
print("Reference source:", network.source)


PNG difference: mean 0.632/255; maximum 12/255
Reference source: {'repository': 'https://github.com/akarshkumar0101/fer', 'commit': '67c270c197d8243435d6a9ad036e26c9c5e3712d', 'genome': 'picbreeder_genomes/skull.zip', 'model': 'evolved Picbreeder CPPN, not SGD'}


## 2. Reproduce the published interventions

Each row varies exactly one original connection weight through offsets −1, −0.5, 0, +0.5, +1. The central image is the baseline. Mouth opening and eye winking are the authors' descriptions; inspect the other changes too, rather than assuming perfect independence.


In [9]:
for name in ("Mouth opening", "Eye winking"):
    spec = network.presets[name]
    record = network.record["presets"][name]
    print(f"{name}: paper parameter {record['parameter_id']}; original connection "
          f"{record['source']} → {record['target']}; inspector connection {spec.target}")
    offsets = [-1, -.5, 0, .5, 1]
    results = sweep(network, spec, [spec.value + delta for delta in offsets])
    display(widgets.HBox([
        widgets.VBox([
            widgets.Label(f"Δw = {delta:+g}" + (" (original)" if delta == 0 else "")),
            widgets.Image(value=png_bytes(result.rgb), width=160),
        ]) for delta, result in zip(offsets, results)
    ], layout=widgets.Layout(flex_flow="row wrap")))


Mouth opening: paper parameter 4371; original connection 540_106 → 576_223; inspector connection (14, 6)


Eye winking: paper parameter 5009; original connection 534_2 → 534_5; inspector connection (-1, 12)


## 3. Explore with the same inspector

Choose a published example, then move the weight slider or run a sweep. The source activation and destination before/after maps explain the local computation. Follow the graph to see subsequent changes and compare them to the final image.

The example selector resets to the original network and sets the paper's ±1 interval with five samples. Every intervention starts from the original; neither the examples nor the manual controls mutate the imported network.

The constant bias input is visible as node −4. To vary an original bias contribution, select a connection from that input. The generic **Node bias** control instead adds an experimental offset; those extra offsets were zero in the imported graph.


In [10]:
inspector = CPPNInspector(network, save_dir=ROOT / "runs" / "skull-interpretability")
example = widgets.Dropdown(options=["Mouth opening", "Eye winking", "Eye width", "Jaw width"],
                           description="Example")

def select_example(change=None):
    spec = network.presets[example.value]
    inspector.kind.value = "weight"
    inspector.target.value = spec.target
    inspector.reset_button.click()
    inspector.low.value = spec.value - 1
    inspector.high.value = spec.value + 1
    inspector.count.value = 5

example.observe(select_example, names="value")
select_example()
display(example, inspector)


Dropdown(description='Example', options=('Mouth opening', 'Eye winking', 'Eye width', 'Jaw width'), value='Mou…

## 4. Record and replay

Use **Save experiment** to save the baseline reference definition, interventions, observation and PNGs in a new directory. Replay does not need the original downloaded files. These experiments reproduce controls in a known evolved network; they do not show that our current breeding procedure discovers comparable representations.


In [ ]:
EXPERIMENT_PATH = None
if EXPERIMENT_PATH is not None:
    original, results = replay_experiment(EXPERIMENT_PATH)
    print(f"Verified baseline and {len(results)} interventions.")


Restart the kernel after changing imported modules. This notebook is independent of notebooks 01–03 and needs no running breeding session, ImageNet classifier, JAX, or SGD training.
